# PARTE 4 - Redes Convolucionales usando FashionMNIST

En esta parte se implementa la Red Neuronal Convolucional (CNN) para clasificar prendas de vestir usando el dataset FashionMNIST.

Las CNN permiten extraer automáticamente características importantes mas detalladas de las imagenes, mejorando el desempeño en tareas de clasificación de imágenes.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

*1 Configuracion*

Se verifica si existe GPU disponible para acelerar el entrenamiento. En caso contrario se utilizara CPU.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)

*2 Carga y preparacion del dataset*

Las imagenes del dataset FashionMNIST  son convertidas a tensores y normalizadas para mejorar la estabilidad del entrenamiento.

La normalizacion evita gradientes inestables, acelera el aprendizaje y mejora la convergencia.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = datasets.FashionMNIST(
    'FashionMNIST_data/',
    train=True,
    download=True,
    transform=transform
)

testset = datasets.FashionMNIST(
    'FashionMNIST_data/',
    train=False,
    download=True,
    transform=transform
)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

print("Entrenamiento:", len(trainset))
print("Prueba:", len(testset))

*3 Visualizar ejemplos*

Aqui mostramos algunas imagenes del dataset junto con susu etiquetas reales 

In [ ]:
images, labels = next(iter(trainloader))

fig, axes = plt.subplots(1,5, figsize=(15,5))

classes = trainset.classes

for i in range(5):
    axes[i].imshow(images[i].squeeze(), cmap='gray')
    axes[i].set_title(labels[i].item())
    axes[i].set_title(classes[labels[i]])
    axes[i].axis("off")

plt.show()


*4 Construccion de la CNN*

La red convolucional permite extraer caracteristicas relevantes y mas detalladas de las imagenes como: 
- Bordes 
- Formas 
- Patrones presentes en las img 

In [ ]:
class CNN_FashionMNIST(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv_layers = nn.Sequential(
            # Conv2d: busca bordes, lineas y formas 
            # ReLU: Introduce no linealidad.
            # MaxPool: Reduce tamaño y elimina información redundante.
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc_layers = nn.Sequential(

            nn.Flatten(), #convierte las características extraídas en un vector.

            nn.Linear(64 * 7 * 7, 128), #Hace la clasificación final.
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):

        x = self.conv_layers(x)
        x = self.fc_layers(x)

        return x


model = CNN_FashionMNIST().to(device)

print(model)

*5 Funcion de perdida y optimizador*

Se utiliza CrossEntropyLoss y el optimizador Adam para ajustar los pesos del modelo.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

*6 Entrenamiento*

In [ ]:
epochs = 10

train_losses = []
accuracies = []

for epoch in range(epochs):

    running_loss = 0

    model.train()

    # ENTRENAMIENTO
    for images, labels in trainloader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    # EVALUACIÓN
    correct = 0
    total = 0

    model.eval()

    with torch.no_grad():

        for images, labels in testloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_loss = running_loss / len(trainloader)

    train_losses.append(avg_loss)
    accuracies.append(accuracy)
    

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.2f}%")
    print("-" * 40)

*7 Graficas*

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(train_losses)
plt.title("Pérdida de entrenamiento")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

plt.figure(figsize=(10,4))
plt.plot(accuracies)
plt.title("Precisión del modelo")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.show()

*8  Guardar modelo y prediccion final*

In [ ]:
torch.save(model.state_dict(), "cnn_fashionmnist.pth")

classes = trainset.classes

images, labels = next(iter(testloader))

model.eval()

with torch.no_grad():

    outputs = model(images.to(device))

    _, predicted = torch.max(outputs, 1)

plt.figure(figsize=(4,4))

plt.imshow(images[0].squeeze(), cmap='gray')

plt.title(
    f"Real: {classes[labels[0].item()]} | Predicción: {classes[predicted[0].item()]}"
)

plt.axis("off")

plt.show()